# Recurrent Neural Networks from Scratch

**Module 05: RNN/LSTM/GRU**  
**Objective**: Understand sequential modeling and backpropagation through time

RNNs process sequences by maintaining hidden state:
- **Memory**: Hidden state $h_t$ carries information from previous steps
- **Recurrence**: $h_t = f(h_{t-1}, x_t)$
- **Challenge**: Vanishing/exploding gradients over long sequences

## What You'll Learn
1. Vanilla RNN implementation
2. Backpropagation Through Time (BPTT)
3. Vanishing gradient problem
4. LSTM architecture (gates solve vanishing gradients)
5. GRU (simplified LSTM)
6. Sequence generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

np.random.seed(42)

## 1. Vanilla RNN

**Forward pass**:
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

**Key properties**:
- Shares weights across time steps
- Hidden state $h_t$ acts as memory
- Can process variable-length sequences

In [ ]:
class VanillaRNN:
    """Simple RNN implementation."""
    
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Initialize weights (Xavier)
        self.Wxh = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
        self.Whh = np.random.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.Why = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
        
        self.losses = []
    
    def forward(self, inputs, h_prev):
        """
        Forward pass through time.
        
        Args:
            inputs: list of (input_size, 1) arrays
            h_prev: (hidden_size, 1) initial hidden state
        
        Returns:
            outputs: list of (output_size, 1) predictions
            hidden_states: list of (hidden_size, 1) hidden states
        """
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = np.copy(h_prev)
        
        # Forward through time
        for t, x in enumerate(inputs):
            xs[t] = x
            hs[t] = np.tanh(self.Wxh @ xs[t] + self.Whh @ hs[t-1] + self.bh)
            ys[t] = self.Why @ hs[t] + self.by
            ps[t] = self.softmax(ys[t])
        
        return xs, hs, ys, ps
    
    def backward(self, xs, hs, ps, targets):
        """
        Backpropagation through time.
        
        Args:
            xs, hs, ps: cached from forward pass
            targets: list of target indices
        
        Returns:
            gradients for all parameters
        """
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        
        dh_next = np.zeros_like(hs[0])
        
        # Backward through time
        for t in reversed(range(len(targets))):
            # Output layer gradient
            dy = np.copy(ps[t])
            dy[targets[t]] -= 1  # Cross-entropy gradient
            
            dWhy += dy @ hs[t].T
            dby += dy
            
            # Hidden layer gradient
            dh = self.Why.T @ dy + dh_next
            
            # Gradient through tanh
            dh_raw = (1 - hs[t] * hs[t]) * dh
            
            dbh += dh_raw
            dWxh += dh_raw @ xs[t].T
            dWhh += dh_raw @ hs[t-1].T
            
            dh_next = self.Whh.T @ dh_raw
        
        # Clip gradients (prevent exploding gradients)
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)
        
        return dWxh, dWhh, dWhy, dbh, dby
    
    def softmax(self, x):
        """Numerically stable softmax."""
        exp_x = np.exp(x - np.max(x))
        return exp_x / np.sum(exp_x)
    
    def sample(self, h, seed_ix, n):
        """
        Generate sequence from model.
        
        Args:
            h: initial hidden state
            seed_ix: seed character index
            n: number of characters to generate
        
        Returns:
            indices: list of sampled character indices
        """
        x = np.zeros((self.input_size, 1))
        x[seed_ix] = 1
        indices = []
        
        for _ in range(n):
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            p = self.softmax(y)
            
            # Sample from distribution
            ix = np.random.choice(range(self.output_size), p=p.ravel())
            
            x = np.zeros((self.input_size, 1))
            x[ix] = 1
            indices.append(ix)
        
        return indices
    
    def train(self, data, char_to_ix, ix_to_char, epochs=100, seq_length=25, learning_rate=0.1):
        """Training loop."""
        
        n, p = 0, 0
        mWxh, mWhh, mWhy = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.Why)
        mbh, mby = np.zeros_like(self.bh), np.zeros_like(self.by)
        
        smooth_loss = -np.log(1.0 / self.output_size) * seq_length
        
        for epoch in range(epochs):
            # Reset memory if needed
            if p + seq_length + 1 >= len(data) or n == 0:
                h_prev = np.zeros((self.hidden_size, 1))
                p = 0
            
            # Prepare inputs and targets
            inputs = [np.zeros((self.input_size, 1)) for _ in range(seq_length)]
            targets = []
            
            for t in range(seq_length):
                inputs[t][char_to_ix[data[p + t]]] = 1
                targets.append(char_to_ix[data[p + t + 1]])
            
            # Forward
            xs, hs, ys, ps = self.forward(inputs, h_prev)
            
            # Compute loss
            loss = 0
            for t in range(len(targets)):
                loss += -np.log(ps[t][targets[t], 0])
            
            smooth_loss = smooth_loss * 0.999 + loss * 0.001
            self.losses.append(smooth_loss)
            
            # Backward
            dWxh, dWhh, dWhy, dbh, dby = self.backward(xs, hs, ps, targets)
            
            # Update with Adagrad
            for param, dparam, mem in zip([self.Wxh, self.Whh, self.Why, self.bh, self.by],
                                          [dWxh, dWhh, dWhy, dbh, dby],
                                          [mWxh, mWhh, mWhy, mbh, mby]):
                mem += dparam * dparam
                param -= learning_rate * dparam / np.sqrt(mem + 1e-8)
            
            # Update
            p += seq_length
            n += 1
            
            # Sample
            if epoch % 100 == 0:
                sample_ix = self.sample(h_prev, char_to_ix[data[0]], 200)
                txt = ''.join(ix_to_char[ix] for ix in sample_ix)
                print(f"\nEpoch {epoch}, Loss: {smooth_loss:.4f}")
                print(f"Sample: {txt[:100]}...")
            
            h_prev = hs[len(inputs) - 1]

# Test on text data
text = "hello world! this is a simple text for testing rnn. " * 20

chars = list(set(text))
data_size, vocab_size = len(text), len(chars)
print(f"Data size: {data_size}, Vocab size: {vocab_size}\n")

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

# Create and train RNN
rnn = VanillaRNN(input_size=vocab_size, hidden_size=100, output_size=vocab_size)

print("Training Vanilla RNN...\n")
rnn.train(text, char_to_ix, ix_to_char, epochs=1000, seq_length=25, learning_rate=0.1)

# Plot loss
plt.figure(figsize=(12, 4))
plt.plot(rnn.losses)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('RNN Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Backpropagation Through Time (BPTT)

**Forward**: Unroll RNN through time steps

**Backward**: Compute gradients by chain rule through time

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial h_t} \frac{\partial h_t}{\partial W_{hh}}$$

**Key insight**: Gradient flows backward through all time steps, leading to vanishing/exploding gradients.

In [ ]:
def visualize_gradient_flow():
    """Visualize vanishing gradient problem."""
    
    # Simulate gradient magnitudes through time
    T = 50  # Time steps
    
    # Different gradient scenarios
    vanishing = np.array([0.9 ** t for t in range(T)])  # < 1
    exploding = np.array([1.1 ** t for t in range(T)])  # > 1
    stable = np.ones(T)  # = 1
    
    plt.figure(figsize=(12, 4))
    
    plt.semilogy(vanishing, label='Vanishing (0.9ᵗ)', marker='o')
    plt.semilogy(exploding, label='Exploding (1.1ᵗ)', marker='s')
    plt.semilogy(stable, label='Stable (1.0)', marker='^')
    
    plt.xlabel('Time Step (backward)')
    plt.ylabel('Gradient Magnitude (log scale)')
    plt.title('Gradient Flow in RNN')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"After 50 steps:")
    print(f"  Vanishing: {vanishing[-1]:.10f} (effectively 0)")
    print(f"  Exploding: {exploding[-1]:.2e} (infinity)")
    print(f"  Stable: {stable[-1]:.2f}")

visualize_gradient_flow()

## 3. LSTM: Long Short-Term Memory

**Solution to vanishing gradients**: Gated cells control information flow

**Three gates**:
1. **Forget gate**: $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ — what to forget
2. **Input gate**: $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ — what to update
3. **Output gate**: $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ — what to output

**Cell state update**:
$$\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
$$h_t = o_t \odot \tanh(C_t)$$

In [ ]:
class LSTMCell:
    """LSTM cell from scratch."""
    
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Combined input size
        combined_size = input_size + hidden_size
        
        # Forget gate
        self.Wf = np.random.randn(hidden_size, combined_size) * 0.01
        self.bf = np.zeros((hidden_size, 1))
        
        # Input gate
        self.Wi = np.random.randn(hidden_size, combined_size) * 0.01
        self.bi = np.zeros((hidden_size, 1))
        
        # Candidate cell state
        self.Wc = np.random.randn(hidden_size, combined_size) * 0.01
        self.bc = np.zeros((hidden_size, 1))
        
        # Output gate
        self.Wo = np.random.randn(hidden_size, combined_size) * 0.01
        self.bo = np.zeros((hidden_size, 1))
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, x, h_prev, c_prev):
        """
        Single time step forward.
        
        Args:
            x: (input_size, 1)
            h_prev: (hidden_size, 1)
            c_prev: (hidden_size, 1) cell state
        
        Returns:
            h: new hidden state
            c: new cell state
            cache: values for backward
        """
        # Concatenate input and previous hidden
        concat = np.vstack((h_prev, x))
        
        # Forget gate
        f = self.sigmoid(self.Wf @ concat + self.bf)
        
        # Input gate
        i = self.sigmoid(self.Wi @ concat + self.bi)
        
        # Candidate cell state
        c_tilde = np.tanh(self.Wc @ concat + self.bc)
        
        # Update cell state
        c = f * c_prev + i * c_tilde
        
        # Output gate
        o = self.sigmoid(self.Wo @ concat + self.bo)
        
        # Hidden state
        h = o * np.tanh(c)
        
        cache = (x, h_prev, c_prev, concat, f, i, c_tilde, c, o)
        
        return h, c, cache

class LSTM:
    """LSTM network."""
    
    def __init__(self, input_size, hidden_size, output_size):
        self.cell = LSTMCell(input_size, hidden_size)
        self.hidden_size = hidden_size
        
        # Output layer
        self.Why = np.random.randn(output_size, hidden_size) * 0.01
        self.by = np.zeros((output_size, 1))
    
    def forward(self, inputs, h0=None, c0=None):
        """Forward through sequence."""
        
        if h0 is None:
            h0 = np.zeros((self.hidden_size, 1))
        if c0 is None:
            c0 = np.zeros((self.hidden_size, 1))
        
        h, c = h0, c0
        outputs = []
        caches = []
        
        for x in inputs:
            h, c, cache = self.cell.forward(x, h, c)
            y = self.Why @ h + self.by
            outputs.append(y)
            caches.append(cache)
        
        return outputs, h, c, caches
    
    def sample(self, seed_ix, n, vocab_size):
        """Generate sequence."""
        
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        
        x = np.zeros((vocab_size, 1))
        x[seed_ix] = 1
        
        indices = []
        
        for _ in range(n):
            h, c, _ = self.cell.forward(x, h, c)
            y = self.Why @ h + self.by
            
            # Softmax
            p = np.exp(y - np.max(y))
            p = p / np.sum(p)
            
            ix = np.random.choice(range(vocab_size), p=p.ravel())
            
            x = np.zeros((vocab_size, 1))
            x[ix] = 1
            indices.append(ix)
        
        return indices

# Test LSTM
print("\nTesting LSTM Cell\n")

lstm = LSTM(input_size=10, hidden_size=20, output_size=10)
test_inputs = [np.random.randn(10, 1) for _ in range(5)]

outputs, h_final, c_final, caches = lstm.forward(test_inputs)

print(f"Input sequence length: {len(test_inputs)}")
print(f"Output sequence length: {len(outputs)}")
print(f"Hidden state shape: {h_final.shape}")
print(f"Cell state shape: {c_final.shape}")
print(f"\n✓ LSTM maintains cell state to combat vanishing gradients!")

## 4. GRU: Gated Recurrent Unit

**Simplified LSTM**: Fewer parameters, similar performance

**Two gates**:
1. **Reset gate**: $r_t = \sigma(W_r [h_{t-1}, x_t])$ — how much past to forget
2. **Update gate**: $z_t = \sigma(W_z [h_{t-1}, x_t])$ — how much to update

**Hidden state update**:
$$\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t])$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

In [ ]:
class GRUCell:
    """GRU cell from scratch."""
    
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        combined_size = input_size + hidden_size
        
        # Reset gate
        self.Wr = np.random.randn(hidden_size, combined_size) * 0.01
        self.br = np.zeros((hidden_size, 1))
        
        # Update gate
        self.Wz = np.random.randn(hidden_size, combined_size) * 0.01
        self.bz = np.zeros((hidden_size, 1))
        
        # Candidate hidden state
        self.Wh = np.random.randn(hidden_size, combined_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, x, h_prev):
        """Single time step."""
        
        concat = np.vstack((h_prev, x))
        
        # Reset gate
        r = self.sigmoid(self.Wr @ concat + self.br)
        
        # Update gate
        z = self.sigmoid(self.Wz @ concat + self.bz)
        
        # Candidate hidden state (with reset applied to h_prev)
        concat_reset = np.vstack((r * h_prev, x))
        h_tilde = np.tanh(self.Wh @ concat_reset + self.bh)
        
        # Hidden state (interpolate between old and new)
        h = (1 - z) * h_prev + z * h_tilde
        
        return h

# Test GRU
print("\nTesting GRU Cell\n")

gru_cell = GRUCell(input_size=10, hidden_size=20)
h = np.zeros((20, 1))

for t in range(5):
    x = np.random.randn(10, 1)
    h = gru_cell.forward(x, h)
    print(f"Step {t+1}: Hidden state shape = {h.shape}")

print(f"\n✓ GRU has fewer parameters than LSTM but similar performance!")

## 5. Comparing RNN, LSTM, GRU

In [ ]:
def compare_architectures():
    """Compare parameter counts."""
    
    input_size = 100
    hidden_size = 200
    
    # RNN parameters
    rnn_params = (
        hidden_size * input_size +  # Wxh
        hidden_size * hidden_size +  # Whh
        hidden_size  # bh
    )
    
    # LSTM parameters (4 gates)
    lstm_params = 4 * (
        hidden_size * (input_size + hidden_size) +  # Weights
        hidden_size  # Bias
    )
    
    # GRU parameters (3 gates, but different structure)
    gru_params = 3 * (
        hidden_size * (input_size + hidden_size) +
        hidden_size
    )
    
    print("\nArchitecture Comparison")
    print("=" * 50)
    print(f"Input size: {input_size}, Hidden size: {hidden_size}\n")
    print(f"{'Architecture':<15} {'Parameters':>15} {'vs RNN':>15}")
    print("=" * 50)
    print(f"{'RNN':<15} {rnn_params:>15,} {1.0:>14.2f}x")
    print(f"{'LSTM':<15} {lstm_params:>15,} {lstm_params/rnn_params:>14.2f}x")
    print(f"{'GRU':<15} {gru_params:>15,} {gru_params/rnn_params:>14.2f}x")
    print("=" * 50)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    
    architectures = ['RNN', 'LSTM', 'GRU']
    params = [rnn_params, lstm_params, gru_params]
    colors = ['skyblue', 'lightcoral', 'lightgreen']
    
    bars = ax.bar(architectures, params, color=colors, edgecolor='black')
    
    # Add values on bars
    for bar, param in zip(bars, params):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{param:,}',
                ha='center', va='bottom', fontsize=12)
    
    ax.set_ylabel('Number of Parameters', fontsize=12)
    ax.set_title('Parameter Comparison: RNN vs LSTM vs GRU', fontsize=14)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_architectures()

## Summary

You've mastered recurrent networks from scratch:
- ✅ Vanilla RNN (forward & BPTT)
- ✅ Vanishing/exploding gradient problem
- ✅ LSTM with gated cells
- ✅ GRU as simplified LSTM
- ✅ Sequence generation

**Key insights**:
1. RNNs process sequences via hidden state
2. BPTT enables learning but causes vanishing gradients
3. LSTM gates control information flow (solve vanishing gradients)
4. GRU is a simpler alternative to LSTM
5. Gradient clipping prevents exploding gradients

**When to use**:
- **RNN**: Short sequences, simple tasks
- **LSTM**: Long sequences, complex dependencies
- **GRU**: Faster training, similar performance to LSTM

**Next Notebook**: LSTMs/GRUs with PyTorch (sentiment analysis, time series forecasting)

## Exercises

1. **Bidirectional RNN**: Implement bidirectional RNN (forward + backward)
2. **Truncated BPTT**: Implement truncated backpropagation for long sequences
3. **Peephole Connections**: Add peephole connections to LSTM (let gates see cell state)